In [ ]:
!pip install azure-storage-blob pandas numpy scikit-learn tensorflow

import os
import json
import pandas as pd
import numpy as np
from io import StringIO
from azure.storage.blob import BlobServiceClient
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tqdm.notebook import tqdm
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from typing import Dict, Any, List
AZURE_CONNECTION_STRING = ""


In [ ]:
import pandas as pd
import numpy as np
import json
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import io
from typing import Dict, Any, List
from concurrent.futures import ThreadPoolExecutor

try:
    from azure.storage.blob import BlobServiceClient, ContainerClient
    from io import BytesIO
except ImportError:
    class BlobServiceClient: pass
    class ContainerClient: pass
    class BytesIO: pass
    print("Warning: azure-storage-blob is not installed. Azure download logic will not execute until installed and credentials are set.")


# --- Project Configuration ---
CSV_FILE_NAME = "ai_rank_gpt.csv"

CONNECTION_STRING = AZURE_CONNECTION_STRING
CONTAINER_NAME = "london"
SEGMENTATION_FOLDER = "SemanticSegmentation/json/" # Ensure trailing slash
DETECTION_ROOT = "YoloV10/"
CONFIDENCE_LEVELS = [0.7]

# Placeholder for the complete set of Image IDs (19995 total)
NUM_IMAGES = 19995
IMAGE_IDS = [f"{i}_{angle}.jpg" for i in range(NUM_IMAGES // 4 + 1) for angle in [0, 90, 180, 270]][:NUM_IMAGES]

SEG_FULL_CLASSES = [
    'Bicycle', 'Bicyclist', 'Bike Lane', 'Bike Rack', 'Billboard', 'Bird',
    'Boat', 'Boat Mount', 'Bridge', 'Building', 'Bus', 'Banner', 'Barrier',
    'Bench', 'Car', 'Car Mount', 'Caravan', 'Catch Basin', 'CCTV Camera',
    'Curb', 'Curb Cut', 'Crosswalk—Plain', 'Ego Vehicle', 'Fence',
    'Fire Hydrant', 'Guard Rail', 'Junction Box', 'Lane', 'Lane Marking—Crosswalk',
    'Lane Marking—General', 'Mailbox', 'Manhole', 'Mountain', 'Motorcycle',
    'Motorcyclist', 'On Rails', 'Other Rider', 'Other Vehicle', 'Parking',
    'Pedestrian Area', 'Phone Booth', 'Pole', 'Pothole', 'Rail Track',
    'Road', 'Sand', 'Service Lane', 'Sidewalk', 'Sky', 'Snow', 'Street Light',
    'Terrain', 'Traffic Light', 'Traffic Sign (Back)', 'Traffic Sign (Front)',
    'Traffic Sign Frame', 'Trailer', 'Trash Can', 'Truck', 'Tunnel', 'Unlabeled',
    'Utility Pole', 'Vegetation', 'Water', 'Wheeled Slow'
]
OBJ_FULL_CLASSES = [
    'Person', 'Bicycle', 'Car', 'Motorcycle', 'Airplane', 'Bus', 'Train', 'Truck', 'Boat', 'Traffic Light',
    'Fire Hydrant', 'Stop Sign', 'Parking Meter', 'Bench', 'Bird', 'Cat', 'Dog', 'Horse', 'Sheep', 'Cow',
    'Elephant', 'Bear', 'Zebra', 'Giraffe', 'Backpack', 'Umbrella', 'Handbag', 'Tie', 'Suitcase', 'Frisbee',
    'Skis', 'Snowboard', 'Sports Ball', 'Kite', 'Baseball Bat', 'Baseball Glove', 'Skateboard', 'Surfboard',
    'Tennis Racket', 'Bottle', 'Wine Glass', 'Cup', 'Fork', 'Knife', 'Spoon', 'Bowl', 'Banana', 'Apple',
    'Sandwich', 'Orange', 'Broccoli', 'Carrot', 'Hot Dog', 'Pizza', 'Donut', 'Cake', 'Chair', 'Couch',
    'Potted Plant', 'Bed', 'Dining Table', 'Toilet', 'Tv', 'Laptop', 'Mouse', 'Remote', 'Keyboard',
    'Cell Phone', 'Microwave', 'Oven', 'Toaster', 'Sink', 'Refrigerator', 'Book', 'Clock', 'Vase',
    'Scissors', 'Teddy Bear', 'Hair Drier', 'Toothbrush'
]

# --- Step 2 Required Class Selection ---
SELECTED_SEG_CLASSES = SEG_FULL_CLASSES
SELECTED_OBJ_CLASSES = OBJ_FULL_CLASSES

# --- Helper Functions for Parsing ---

def parse_segmentation(json_content: str) -> Dict[str, float]:
    """Parses segmentation JSON content and returns a dictionary of class names to pixel percentages."""
    try:
        data = json.loads(json_content)
        features = {}
        # Assuming a fixed image size like 640x640, or use the provided dimensions
        total_pixels = data.get('height', 640) * data.get('width', 640)

        for entry in data.get('label_histogram', []):
            # Store pixel percentage relative to the image size
            features[entry['label_name']] = entry['pixels'] / total_pixels
        return features
    except Exception as e:
        print(f"Error parsing segmentation JSON: {e}")
        return {}

def parse_yolo(txt_content: str) -> Dict[str, int]:
    """Parses YOLO TXT content and returns a dictionary of object class counts."""
    try:
        lines = txt_content.strip().split('\n')
        class_counts = {}
        for line in lines:
            parts = line.split()
            if parts:
                # YOLO format is typically: class_id confidence x_center y_center width height
                class_id = int(parts[0])
                # We map the ID to the class name using the full list
                if class_id < len(OBJ_FULL_CLASSES):
                    class_name = OBJ_FULL_CLASSES[class_id]
                    class_counts[class_name] = class_counts.get(class_name, 0) + 1
        return class_counts
    except Exception as e:
        print(f"Error parsing YOLO TXT: {e}")
        return {}

MAX_WORKERS = 32

def download_and_process_blob(blob_item, container_client, is_segmentation: bool):
    """Downloads, reads, and parses a single blob item. Returns (image_id, features) or None on error."""
    try:
        blob_name = blob_item['name'] # Use the name from the blob dictionary/item

        blob_client = container_client.get_blob_client(blob_name)
        download_stream = blob_client.download_blob()
        content = download_stream.readall().decode('utf-8')

        if is_segmentation:
            if not blob_name.endswith('.json'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_segmentation(content)
            prefixed_features = {f"seg_{k}": v for k, v in features.items()}
        else:
            if not blob_name.endswith('.txt'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_yolo(content)
            prefixed_features = {f"obj_{k}": v for k, v in features.items()}

        if image_id not in IMAGE_IDS:
            return None

        return (image_id, prefixed_features)

    except Exception as e:
        return None

# --- Step 1: Download and Combine Logic ---

def download_and_combine_data() -> Dict[str, pd.DataFrame]:
    """
    Connects to Azure Blob Storage, downloads segmentation and object detection
    data for all confidence levels using parallel processing, and combines them.
    """
    print("--- Step 1: Connecting to Azure and Downloading All Blobs (Parallelized) ---")

    if CONNECTION_STRING == "YOUR_AZURE_BLOB_STORAGE_CONNECTION_STRING_HERE":
         raise ValueError("Please replace 'YOUR_AZURE_BLOB_STORAGE_CONNECTION_STRING_HERE' with your actual Azure connection string.")

    try:
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)
    except Exception as e:
        raise ConnectionError(f"Failed to connect to Azure: {e}")

    # Initialize a master map to store all features for all images
    feature_map = {image_id: {} for image_id in IMAGE_IDS}

    # 1. Process Segmentation Data (Consistent across all confidence levels)
    print("\n[1/2] Processing Semantic Segmentation Data (JSON files)...")

    # Efficiently list all relevant segmentation blobs
    seg_blob_items = list(container_client.list_blobs(name_starts_with=SEGMENTATION_FOLDER, results_per_page=1000).by_page())
    seg_blob_items = [blob for page in seg_blob_items for blob in page] # Flatten list of pages

    seg_tasks = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for blob_item in seg_blob_items:
            seg_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, True))

        for future in tqdm(seg_tasks, desc="Downloading Segmentation (Parallel)"):
            result = future.result()
            if result:
                image_id, features = result
                feature_map[image_id].update(features)


    # 2. Process Object Detection Data for all 3 confidence levels
    all_dataframes = {}

    print("\n[2/2] Processing Object Detection (YOLO) Data (TXT files) for 3 Confidence Levels...")

    for conf in CONFIDENCE_LEVELS:
        conf_str = f"{conf:.1f}"
        detection_folder = f"{DETECTION_ROOT}conf_{conf_str}/labels/"

        # List all relevant detection blobs
        obj_blob_items = list(container_client.list_blobs(name_starts_with=detection_folder, results_per_page=1000).by_page())
        obj_blob_items = [blob for page in obj_blob_items for blob in page]

        current_conf_features = {img_id: feature_map[img_id].copy() for img_id in IMAGE_IDS}

        obj_tasks = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for blob_item in obj_blob_items:
                obj_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, False))

            for future in tqdm(obj_tasks, desc=f"Downloading YOLO {conf_str} (Parallel)"):
                result = future.result()
                if result:
                    image_id, features = result
                    current_conf_features[image_id].update(features)


        # Convert the map for this confidence level into a DataFrame
        data_rows = []
        for img_id, features in current_conf_features.items():
            row = {'image_name': img_id}
            row.update(features)
            data_rows.append(row)

        df = pd.DataFrame(data_rows).set_index('image_name')

        # Define all expected columns for consistent zero-filling
        seg_cols_full = [f"seg_{cls}" for cls in SEG_FULL_CLASSES]
        obj_cols_full = [f"obj_{cls}" for cls in OBJ_FULL_CLASSES]
        all_cols_full = seg_cols_full + obj_cols_full

        # Ensure all possible columns exist and fill NaN (from missing entries) with 0.0
        df = df.reindex(columns=all_cols_full, fill_value=0.0)

        all_dataframes[f'conf_{conf_str}'] = df
        print(f"DataFrame for confidence {conf_str} created. Shape: {df.shape}")

    return all_dataframes


# --- Step 2: Process Feature Selection ---

def process_feature_selection(data_frames: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Filters the dataframes to only keep the most relevant, pre-selected classes."""
    print("\n--- Step 2: Feature Selection ---")

    # Now using ALL available classes as features
    selected_seg_cols = [f"seg_{cls}" for cls in SELECTED_SEG_CLASSES]
    selected_obj_cols = [f"obj_{cls}" for cls in SELECTED_OBJ_CLASSES]
    KEEP_COLS = selected_seg_cols + selected_obj_cols

    processed_dfs = {}
    for name, df in data_frames.items():
        filtered_df = df[KEEP_COLS].copy()
        processed_dfs[name] = filtered_df
        print(f"DataFrame {name} filtered. Final Feature Count: {filtered_df.shape[1]}")

    return processed_dfs

# --- Step 3: TrueSkill Processing and Safety Binning ---

def process_trueskill_and_binning(data_frames: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Loads TrueSkill CSV, calculates 10 safety bins, and merges the target variable."""
    print("\n--- Step 3: TrueSkill Processing and Safety Binning ---")

    ts_df = pd.read_csv(CSV_FILE_NAME)
    ts_df = ts_df.set_index('image_name')

    # Use 'ai_ts_score' as the ranking metric. Higher score is safer/better.
    ts_df = ts_df[['ai_ts_score']].rename(columns={'ai_ts_score': 'safety_score'})

    # Create 10 Safety Bins (Target Variable)
    # pd.qcut ensures ~equal samples in each bin (equal-depth binning).
    # Since higher TS score is safer, the labels 1 to 10 will correctly map
    # 1=least safe (lowest scores) and 10=most safe (highest scores).
    ts_df['safety_bin_1_10'] = pd.qcut(
        ts_df['safety_score'],
        q=10,
        labels=False,
        duplicates='drop'
    )
    # Shift labels from 0-9 to 1-10
    ts_df['safety_bin_1_10'] = ts_df['safety_bin_1_10'] + 1

    print(f"TrueSkill data loaded and binned. Total unique images: {len(ts_df)}")
    print("Safety Bin Distribution (1=Unsafe, 10=Safest):")
    print(ts_df['safety_bin_1_10'].value_counts().sort_index())

    # Merge the target variable onto the feature dataframes
    final_dfs = {}
    for name, df in data_frames.items():
        # Inner merge only keeps images present in both features and TrueSkill data
        merged_df = df.merge(ts_df[['safety_bin_1_10']], left_index=True, right_index=True, how='inner')
        final_dfs[name] = merged_df
        print(f"Merged {name}. Final Shape: {merged_df.shape}")

    return final_dfs

# --- Step 4: Signal Analysis ---

def perform_signal_analysis(df: pd.DataFrame) -> None:
    """Performs preliminary analysis to check for signal relevance and provides a report."""
    print("\n--- Step 4: Preliminary Signal Analysis (Confidence 0.7 Data) ---")

    # 1. Feature-Target Correlation
    target_correlation = df.corr(numeric_only=True)['safety_bin_1_10'].drop('safety_bin_1_10').sort_values(ascending=False)

    print("\nTop 10 Feature Correlations with Safety Bin (Absolute Value):")
    print(target_correlation.abs().nlargest(10).to_string())

    # 2. Feature Distribution across Bins (Visualization)
    top_features = target_correlation.abs().nlargest(4).index.tolist()

    # Prepare data for plotting (group by bin and calculate mean)
    plot_data = df.groupby('safety_bin_1_10')[top_features].mean().reset_index()
    plot_data_melted = plot_data.melt(id_vars='safety_bin_1_10', var_name='Feature', value_name='Mean Value')

    # Create the plot
    plt.figure(figsize=(12, 8))
    sns.lineplot(data=plot_data_melted, x='safety_bin_1_10', y='Mean Value', hue='Feature', marker='o', palette='viridis')
    plt.title('Mean Feature Value Across 10 Safety Bins (1=Unsafe, 10=Safest)')
    plt.xlabel('Safety Bin (1 to 10)')
    plt.ylabel('Mean Feature Value (Pixel % or Object Count)')
    plt.xticks(plot_data['safety_bin_1_10'])
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(title='Feature')
    plt.tight_layout()
    plt.show()

    # --- LLM Analysis Summary ---
    print("\n")
    print("\n=======================================================")
    print("      Preliminary Signal Analysis Report (0.7 Conf)")
    print("=======================================================")

# --- Main Execution Block ---

def main():
    try:
        # Step 1: Download and Combine Data
        all_combined_dfs = download_and_combine_data()

        # Step 2: Feature Selection
        selected_dfs = process_feature_selection(all_combined_dfs)

        # Step 3: Target Binning and Merge
        all_final_dfs = process_trueskill_and_binning(selected_dfs)

        # Save the processed dataframes locally (e.g., in Colab's file system)
        for name, df in all_final_dfs.items():
            save_path = f"processed_features_{name}.csv"
            df.to_csv(save_path)
            print(f"\nFinal feature file saved: {save_path} (Shape: {df.shape})")

        # Step 4: Signal Analysis (using the 0.7 confidence data as the primary feature set)
        perform_signal_analysis(all_final_dfs['conf_0.7'])

    except FileNotFoundError as e:
        print(f"\nFATAL ERROR: {e}")
        print("Please ensure your uploaded file (ai_rank_gpt.csv) is accessible.")
    except ConnectionError as e:
        print(f"\nFATAL ERROR: {e}")
        print("Please check your Azure CONNECTION_STRING and ensure the required libraries are installed.")
    except ValueError as e:
        print(f"\nFATAL ERROR: {e}")
        print("You must update the Azure CONNECTION_STRING placeholder.")
    except Exception as e:
        print(f"\nAn unexpected error occurred during the pipeline execution: {e}")

main()

In [ ]:
# Prerequisite Installation in Colab:
# !pip install pandas numpy scikit-learn tensorflow
# The script assumes these libraries are available.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
# Import for signal analysis and evaluation
from sklearn.metrics import classification_report
import os

# --- Configuration ---
# Use the 0.7 confidence feature set as the base for training
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
RANDOM_STATE = 42
TOTAL_CLASSES = 3

print(f"--- Starting NN Data Preparation using {FEATURE_FILE} ---")

# 1. Load Data and Ensure NaNs are Handled
try:
    # Load the processed features (X)
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name')

    # Load the TrueSkill scores for the target (y)
    ts_df = pd.read_csv(TRUESKILL_FILE)
    ts_df = ts_df.set_index('image_name')
    ts_df = ts_df[['ai_ts_score']].rename(columns={'ai_ts_score': 'safety_score'})

except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Required file not found. Ensure '{FEATURE_FILE}' and '{TRUESKILL_FILE}' exist.")
    raise e

# --- Safety Check: NaN Filling ---
X_df = X_df.fillna(0.0)
print(f"Feature DataFrame loaded (Shape: {X_df.shape}). NaN values successfully filled with 0.0.")

# 2. Target Variable (Safety Bin) Preparation
# Re-calculate the 10 safety bins to ensure consistency
minv = ts_df['safety_score'].min()
maxv = ts_df['safety_score'].max()
bins = np.linspace(minv, maxv, TOTAL_CLASSES + 1)   # 11 edges -> 10 equal-width bins

ts_df['safety_bin_1_10'] = pd.cut(
    ts_df['safety_score'],
    bins=bins,
    labels=range(1, TOTAL_CLASSES + 1),    # labels 1..10
    include_lowest=True
).astype(int)
counts = ts_df['safety_bin_1_10'].value_counts(sort=False).reindex(range(1,TOTAL_CLASSES + 1), fill_value=0)
print("Counts per equal-width safety bin (1..10):")
print(counts)

# 3. print bin ranges so you know what each label means
print("\nLabel -> numeric range (inclusive left, exclusive right except last):")
for i in range(10):
    left, right = bins[i], bins[i+1]
    # close the last interval on the right for readability
    right_bracket = ']' if i == 9 else ')'
    print(f"Label {i+1}: [{left:.6g}, {right:.6g}{right_bracket}")

# 4. bar plot (histogram of counts per label)
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Safety bin (equal-width, 1 = lowest range)')
ax.set_ylabel('Number of features')
ax.set_title('Number of features in each equal-width safety bin (1..10)')
ax.set_xticklabels([str(i) for i in range(1,TOTAL_CLASSES + 1)], rotation=0)
plt.tight_layout()
plt.show()

# 3. Final Merge (Inner Join to match images present in both feature and target files)
# Get the target series
target_bins = ts_df['safety_bin_1_10']

# Merge the features DataFrame with the target Series
# This merge is what causes the '_x' (from X_df) and '_y' (from target_bins) suffixes.
combined_df = X_df.merge(target_bins, left_index=True, right_index=True, how='inner')

# Separate Features (X) and Target (y)
TARGET_COL_Y = 'safety_bin_1_10_y'
TARGET_COL_X = 'safety_bin_1_10_x'

# MODIFICATION START: Use the known suffixed column names.
if TARGET_COL_Y in combined_df.columns:
    # The suffixes are present. Use the '_y' column for the target (the one derived in this script).
    X_COLUMNS_TO_DROP = [TARGET_COL_X, TARGET_COL_Y]

    # Extract the target variable
    y_raw = combined_df[TARGET_COL_Y].values - 1 # Use 0-9 indices for Keras to_categorical

    # Drop both target columns from the features
    X = combined_df.drop(columns=X_COLUMNS_TO_DROP).values

# Check for the clean column name, in case the previous script's output file was fixed manually
elif 'safety_bin_1_10' in combined_df.columns:
    TARGET_COL_CLEAN = 'safety_bin_1_10'
    X = combined_df.drop(columns=[TARGET_COL_CLEAN]).values
    y_raw = combined_df[TARGET_COL_CLEAN].values - 1
else:
    # This block provides the same error message but is now an exhaustive final check.
    print(f"ERROR: Failed to find a usable target column. Available columns:")
    print(combined_df.columns.tolist())
    raise KeyError("Could not find 'safety_bin_1_10', 'safety_bin_1_10_y', or a valid target column.")

# --- Data Preparation for Neural Network ---

# 4. Feature Scaling (Standardization is essential for NNs)
print("\n[4/6] Scaling features using StandardScaler...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. One-Hot Encoding the Target
# The target is now 10 classes (0 through 9)
print("[5/6] One-Hot Encoding the target variable (10 bins)...")
y_encoded = to_categorical(y_raw, num_classes=10)

# 6. Train/Test Split
# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

# --- Summary of Prepared Data ---
print("\n--- Summary of Prepared Data for Neural Network ---")
print(f"Number of Features (Input Dimension): {X_train.shape[1]}")
print(f"Number of Classes (Output Dimension): {y_train.shape[1]}")
print(f"Training Features (X_train) Shape: {X_train.shape}")
print(f"Training Target (y_train) Shape: {y_train.shape}")
print(f"Testing Features (X_test) Shape: {X_test.shape}")
print(f"Testing Target (y_test) Shape: {y_test.shape}")


# --- Simple Model Definition and Check ---
# Define a simple NN to verify the data shapes and setup are correct
def create_model(input_dim, output_dim):
    """Creates a simple feed-forward neural network for classification."""
    model = Sequential([
        # First layer with ReLU activation and L2 regularization to prevent initial overfitting
        Dense(128, activation='relu', input_shape=(input_dim,)),
        Dropout(0.2),
        # Hidden layer
        Dense(64, activation='relu'),
        # Output layer: 10 classes, softmax for multi-class probability output
        Dense(output_dim, activation='softmax')
    ])

    # Compile the model with an optimizer and loss function suitable for multi-class classification
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create and summarize the model
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]
model = create_model(input_dim, output_dim)

print("\n[6/6] Simple Neural Network Model Summary (Ready for Training):")
model.summary()

# Dummy training run (10 epochs) to confirm the signal strength
print("\n--- Training Model for Signal Analysis (10 Epochs) ---")
history = model.fit(
    X_train, y_train,
    epochs=10, # Increased from 1 to 10 for better signal assessment
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# --- Signal Strength Analysis ---
print("\n--- Signal Strength Analysis via Classification Report ---")

# 1. Make predictions on the test set
y_pred_probs = model.predict(X_test)

# 2. Convert probabilities and one-hot true values back to class labels (0-9)
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

# 3. Generate Classification Report
# Target names are adjusted from 0-9 back to 1-10 for readability
target_names = [f'Bin {i}' for i in range(1, 11)]

# FIX: Explicitly pass the full range of labels (0-9) to the report to prevent ValueError
# when some bins are missing from the small test set.
print(classification_report(y_test_labels, y_pred_labels, target_names=target_names, labels=range(10), zero_division=0))

print("\n--- Analysis Complete ---")
print("The Classification Report above is the key indicator of signal strength.")

In [ ]:
# Human votes

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# Import for class weight calculation and signal analysis
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import os

# --- Configuration ---
# NOTE: The script assumes FEATURE_FILE and TRUESKILL_FILE exist in the current directory
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "human_trueskill_rankings.csv"
RANDOM_STATE = 42
TOTAL_CLASSES = 10

print(f"--- Starting Robust NN Training using {FEATURE_FILE} ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['human_ts_score']].rename(columns={'human_ts_score': 'safety_score'})
except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Required file not found. Ensure '{FEATURE_FILE}' and '{TRUESKILL_FILE}' exist and are loaded.")
    raise e

print(f"Feature DataFrame loaded (Shape: {X_df.shape}). NaN values successfully filled with 0.0.")

# 2. Target Variable (Safety Bin) Preparation
# Re-calculate the 10 safety bins to ensure consistency
minv = ts_df['safety_score'].min()
maxv = ts_df['safety_score'].max()
bins = np.linspace(minv, maxv, 11)   # 11 edges -> 10 equal-width bins

ts_df['safety_bin_1_10'] = pd.cut(
    ts_df['safety_score'],
    bins=bins,
    labels=range(1, 11),    # labels 1..10
    include_lowest=True
).astype(int)
counts = ts_df['safety_bin_1_10'].value_counts(sort=False).reindex(range(1,11), fill_value=0)
print("Counts per equal-width safety bin (1..10):")
print(counts)

# 3. print bin ranges so you know what each label means
print("\nLabel -> numeric range (inclusive left, exclusive right except last):")
for i in range(10):
    left, right = bins[i], bins[i+1]
    # close the last interval on the right for readability
    right_bracket = ']' if i == 9 else ')'
    print(f"Label {i+1}: [{left:.6g}, {right:.6g}{right_bracket}")

# 4. bar plot (histogram of counts per label)
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Safety bin (equal-width, 1 = lowest range)')
ax.set_ylabel('Number of features')
ax.set_title('Number of features in each equal-width safety bin (1..10)')
ax.set_xticklabels([str(i) for i in range(1,11)], rotation=0)
plt.tight_layout()
plt.show()

# min_score = ts_df['safety_score'].min()
# max_score = ts_df['safety_score'].max()
# bins = np.linspace(min_score, max_score, TOTAL_CLASSES + 1)
# ts_df['safety_bin_1_10'] = pd.cut(
#     ts_df['safety_score'],
#     bins=bins,
#     labels=False,
#     include_lowest=True
# )
# The y_raw_full series contains the 0-9 labels indexed by image_name
y_raw_full = ts_df['safety_bin_1_10']

# 3. Final Merge and Alignment
combined_df = X_df.merge(y_raw_full, left_index=True, right_index=True, how='inner')

# Find the target column name
target_col_name = [col for col in combined_df.columns if 'safety_bin_1_10' in col][-1]

X = combined_df.drop(columns=[target_col_name]).values
y_raw = combined_df[target_col_name].values
y_raw = y_raw.astype(int) - 1
# 4. Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\n[4/8] Features scaled using StandardScaler.")

# --- Data Split: 80% Train, 10% Validation, 10% Test ---

# 5. Split 1: 80% Train, 20% Temp (Test + Validation)
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_raw
)

# 6. Split 2: Divide Temp (20%) into Test (10%) and Validation (10%) (0.5 means 50% of 20% = 10% total)
X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_raw
)

# 7. One-Hot Encoding the Target Sets
y_train = to_categorical(y_train_raw, num_classes=TOTAL_CLASSES)
y_val = to_categorical(y_val_raw, num_classes=TOTAL_CLASSES)
y_test = to_categorical(y_test_raw, num_classes=TOTAL_CLASSES)
print("[7/8] Target variables One-Hot Encoded.")


# 8. Calculate Class Weights (Crucial for Imbalanced Data)
classes = np.unique(y_train_raw)
class_weights_array = compute_class_weight(
    'balanced',
    classes=classes,
    y=y_train_raw
)
class_weight_dict = dict(zip(classes, class_weights_array))
print("\n[8/8] Class weights calculated to handle data imbalance:")
print(class_weight_dict)

# --- Summary of Prepared Data ---
print("\n--- Summary of Prepared Data for Robust Training ---")
print(f"Training Features (X_train) Shape: {X_train.shape} (80%)")
print(f"Validation Features (X_val) Shape: {X_val.shape} (10%)")
print(f"Testing Features (X_test) Shape: {X_test.shape} (10%)")


# --- Robust Model Definition ---
def create_robust_model(input_dim, output_dim):
    """Creates a deeper, more robust feed-forward neural network."""
    model = Sequential([
        # Input Layer: Wider layer to capture initial feature interactions
        Dense(256, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        # Hidden Layer 1: Increased depth
        Dense(128, activation='relu'),
        Dropout(0.3),
        # Hidden Layer 2
        Dense(64, activation='relu'),
        # Output layer
        Dense(output_dim, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create and summarize the model
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]
model = create_robust_model(input_dim, output_dim)

print("\n--- Robust Neural Network Model Summary ---")
model.summary()

# --- Callbacks for Robust Training ---
# Early Stopping: Stop training when validation loss stops improving (patience=10 epochs)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

# ReduceLROnPlateau: Reduce learning rate if val_loss plateaus
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

callbacks = [early_stop, reduce_lr]

# --- Training the Robust Model ---
print("\n--- Starting Robust Model Training (Max 100 Epochs) ---")
history = model.fit(
    X_train, y_train,
    epochs=100, # Max epochs set high, relying on EarlyStopping
    batch_size=32,
    validation_data=(X_val, y_val), # Use the dedicated validation set
    class_weight=class_weight_dict, # Apply calculated weights for imbalance
    callbacks=callbacks,
    verbose=1
)

# --- Final Evaluation on the Test Set ---
print("\n--- Final Evaluation on Dedicated 10% Test Set ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# --- Comprehensive Classification Report ---
print("\n--- Comprehensive Classification Report (Test Set) ---")
y_pred_probs = model.predict(X_test)
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

target_names = [f'Bin {i}' for i in range(1, TOTAL_CLASSES + 1)]
# Set zero_division=0 to handle cases where a bin has no predictions/true samples
print(classification_report(y_test_labels, y_pred_labels, target_names=target_names, labels=range(TOTAL_CLASSES), zero_division=0))

# --- PLOTTING THE CONFUSION MATRIX  ---
print("\n--- Plotting Confusion Matrix ---")

# Calculate the confusion matrix
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=range(TOTAL_CLASSES))

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(TOTAL_CLASSES, TOTAL_CLASSES))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix (Test Set)', fontsize=16)
plt.show()

# --- PLOTTING THE TRAINING CURVES  ---
print("\n--- Generating Training and Validation Curves ---")

# Extract metrics from the history object
train_acc = history.history['accuracy']
train_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
epochs = range(1, len(train_acc) + 1)

# Find the best epoch based on minimum validation loss (for plotting marker)
best_epoch_index = np.argmin(val_loss)
best_epoch = best_epoch_index + 1

plt.figure(figsize=(14, 6))

# Subplot 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, train_acc, 'go-', label='Training Accuracy', linewidth=2, alpha=0.8)
plt.plot(epochs, val_acc, 'bo-', label='Validation Accuracy', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_acc[best_epoch_index], 'r*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Accuracy', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# Subplot 2: Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, train_loss, 'r^-', label='Training Loss', linewidth=2, alpha=0.8)
plt.plot(epochs, val_loss, 'c^-', label='Validation Loss', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_loss[best_epoch_index], 'm*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Loss', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (Categorical Crossentropy)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

print("\n--- Robust Training and Evaluation Complete ---")
print("Check the Test Accuracy and the Classification Report. The class weights should now help improve performance in the previously ignored bins.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# Import for class weight calculation and signal analysis
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import os

# --- Configuration ---
# NOTE: The script assumes FEATURE_FILE and TRUESKILL_FILE exist in the current directory
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
RANDOM_STATE = 42
TOTAL_CLASSES = 10

print(f"--- Starting Robust NN Training using {FEATURE_FILE} ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['ai_ts_score']].rename(columns={'ai_ts_score': 'safety_score'})
except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Required file not found. Ensure '{FEATURE_FILE}' and '{TRUESKILL_FILE}' exist and are loaded.")
    raise e

print(f"Feature DataFrame loaded (Shape: {X_df.shape}). NaN values successfully filled with 0.0.")

# 2. Target Variable (Safety Bin) Preparation
# Re-calculate the 10 safety bins to ensure consistency
minv = ts_df['safety_score'].min()
maxv = ts_df['safety_score'].max()
bins = np.linspace(minv, maxv, 11)   # 11 edges -> 10 equal-width bins

ts_df['safety_bin_1_10'] = pd.cut(
    ts_df['safety_score'],
    bins=bins,
    labels=range(1, 11),    # labels 1..10
    include_lowest=True
).astype(int)
counts = ts_df['safety_bin_1_10'].value_counts(sort=False).reindex(range(1,11), fill_value=0)
print("Counts per equal-width safety bin (1..10):")
print(counts)

# 3. print bin ranges so you know what each label means
print("\nLabel -> numeric range (inclusive left, exclusive right except last):")
for i in range(10):
    left, right = bins[i], bins[i+1]
    # close the last interval on the right for readability
    right_bracket = ']' if i == 9 else ')'
    print(f"Label {i+1}: [{left:.6g}, {right:.6g}{right_bracket}")

# 4. bar plot (histogram of counts per label)
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Safety bin (equal-width, 1 = lowest range)')
ax.set_ylabel('Number of features')
ax.set_title('Number of features in each equal-width safety bin (1..10)')
ax.set_xticklabels([str(i) for i in range(1,11)], rotation=0)
plt.tight_layout()
plt.show()

# min_score = ts_df['safety_score'].min()
# max_score = ts_df['safety_score'].max()
# bins = np.linspace(min_score, max_score, TOTAL_CLASSES + 1)
# ts_df['safety_bin_1_10'] = pd.cut(
#     ts_df['safety_score'],
#     bins=bins,
#     labels=False,
#     include_lowest=True
# )
# The y_raw_full series contains the 0-9 labels indexed by image_name
y_raw_full = ts_df['safety_bin_1_10']

# 3. Final Merge and Alignment
combined_df = X_df.merge(y_raw_full, left_index=True, right_index=True, how='inner')

# Find the target column name
target_col_name = [col for col in combined_df.columns if 'safety_bin_1_10' in col][-1]

X = combined_df.drop(columns=[target_col_name]).values
y_raw = combined_df[target_col_name].values
y_raw = y_raw.astype(int) - 1
# 4. Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\n[4/8] Features scaled using StandardScaler.")

# --- Data Split: 80% Train, 10% Validation, 10% Test ---

# 5. Split 1: 80% Train, 20% Temp (Test + Validation)
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_raw
)

# 6. Split 2: Divide Temp (20%) into Test (10%) and Validation (10%) (0.5 means 50% of 20% = 10% total)
X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_raw
)

# 7. One-Hot Encoding the Target Sets
y_train = to_categorical(y_train_raw, num_classes=TOTAL_CLASSES)
y_val = to_categorical(y_val_raw, num_classes=TOTAL_CLASSES)
y_test = to_categorical(y_test_raw, num_classes=TOTAL_CLASSES)
print("[7/8] Target variables One-Hot Encoded.")


# 8. Calculate Class Weights (Crucial for Imbalanced Data)
classes = np.unique(y_train_raw)
class_weights_array = compute_class_weight(
    'balanced',
    classes=classes,
    y=y_train_raw
)
class_weight_dict = dict(zip(classes, class_weights_array))
print("\n[8/8] Class weights calculated to handle data imbalance:")
print(class_weight_dict)

# --- Summary of Prepared Data ---
print("\n--- Summary of Prepared Data for Robust Training ---")
print(f"Training Features (X_train) Shape: {X_train.shape} (80%)")
print(f"Validation Features (X_val) Shape: {X_val.shape} (10%)")
print(f"Testing Features (X_test) Shape: {X_test.shape} (10%)")


# --- Robust Model Definition ---
def create_robust_model(input_dim, output_dim):
    """Creates a deeper, more robust feed-forward neural network."""
    model = Sequential([
        # Input Layer: Wider layer to capture initial feature interactions
        Dense(256, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        # Hidden Layer 1: Increased depth
        Dense(128, activation='relu'),
        Dropout(0.3),
        # Hidden Layer 2
        Dense(64, activation='relu'),
        # Output layer
        Dense(output_dim, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create and summarize the model
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]
model = create_robust_model(input_dim, output_dim)

print("\n--- Robust Neural Network Model Summary ---")
model.summary()

# --- Callbacks for Robust Training ---
# Early Stopping: Stop training when validation loss stops improving (patience=10 epochs)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

# ReduceLROnPlateau: Reduce learning rate if val_loss plateaus
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

callbacks = [early_stop, reduce_lr]

# --- Training the Robust Model ---
print("\n--- Starting Robust Model Training (Max 100 Epochs) ---")
history = model.fit(
    X_train, y_train,
    epochs=100, # Max epochs set high, relying on EarlyStopping
    batch_size=32,
    validation_data=(X_val, y_val), # Use the dedicated validation set
    class_weight=class_weight_dict, # Apply calculated weights for imbalance
    callbacks=callbacks,
    verbose=1
)

# --- Final Evaluation on the Test Set ---
print("\n--- Final Evaluation on Dedicated 10% Test Set ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# --- Comprehensive Classification Report ---
print("\n--- Comprehensive Classification Report (Test Set) ---")
y_pred_probs = model.predict(X_test)
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

target_names = [f'Bin {i}' for i in range(1, TOTAL_CLASSES + 1)]
# Set zero_division=0 to handle cases where a bin has no predictions/true samples
print(classification_report(y_test_labels, y_pred_labels, target_names=target_names, labels=range(TOTAL_CLASSES), zero_division=0))

# --- PLOTTING THE CONFUSION MATRIX  ---
print("\n--- Plotting Confusion Matrix ---")

# Calculate the confusion matrix
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=range(TOTAL_CLASSES))

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(TOTAL_CLASSES, TOTAL_CLASSES))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix (Test Set)', fontsize=16)
plt.show()

# --- PLOTTING THE TRAINING CURVES  ---
print("\n--- Generating Training and Validation Curves ---")

# Extract metrics from the history object
train_acc = history.history['accuracy']
train_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
epochs = range(1, len(train_acc) + 1)

# Find the best epoch based on minimum validation loss (for plotting marker)
best_epoch_index = np.argmin(val_loss)
best_epoch = best_epoch_index + 1

plt.figure(figsize=(14, 6))

# Subplot 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, train_acc, 'go-', label='Training Accuracy', linewidth=2, alpha=0.8)
plt.plot(epochs, val_acc, 'bo-', label='Validation Accuracy', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_acc[best_epoch_index], 'r*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Accuracy', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# Subplot 2: Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, train_loss, 'r^-', label='Training Loss', linewidth=2, alpha=0.8)
plt.plot(epochs, val_loss, 'c^-', label='Validation Loss', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_loss[best_epoch_index], 'm*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Loss', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (Categorical Crossentropy)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

print("\n--- Robust Training and Evaluation Complete ---")
print("Check the Test Accuracy and the Classification Report. The class weights should now help improve performance in the previously ignored bins.")

In [ ]:
# Cross validation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import backend as K
import os

# --- Configuration ---
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
RANDOM_STATE = 42
TOTAL_CLASSES = 10
N_SPLITS = 5

print(f"--- Starting Robust NN 5-Fold CV using {FEATURE_FILE} ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['ai_ts_score']].rename(columns={'ai_ts_score': 'safety_score'})
except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Required file not found. Ensure '{FEATURE_FILE}' and '{TRUESKILL_FILE}' exist and are loaded.")
    raise e

print(f"Feature DataFrame loaded (Shape: {X_df.shape}). NaN values filled with 0.0.")

# 2. Target Variable (Safety Bin) Preparation
minv = ts_df['safety_score'].min()
maxv = ts_df['safety_score'].max()
bins = np.linspace(minv, maxv, 11)   # 11 edges -> 10 equal-width bins

ts_df['safety_bin_1_10'] = pd.cut(
    ts_df['safety_score'],
    bins=bins,
    labels=range(1, 11),    # labels 1..10
    include_lowest=True
).astype(int)
counts = ts_df['safety_bin_1_10'].value_counts(sort=False).reindex(range(1,11), fill_value=0)
print("Counts per equal-width safety bin (1..10):")
print(counts)

# 3. print bin ranges so you know what each label means
print("\nLabel -> numeric range (inclusive left, exclusive right except last):")
for i in range(10):
    left, right = bins[i], bins[i+1]
    # close the last interval on the right for readability
    right_bracket = ']' if i == 9 else ')'
    print(f"Label {i+1}: [{left:.6g}, {right:.6g}{right_bracket}")

# 4. bar plot (histogram of counts per label)
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Safety bin (equal-width, 1 = lowest range)')
ax.set_ylabel('Number of features')
ax.set_title('Number of features in each equal-width safety bin (1..10)')
ax.set_xticklabels([str(i) for i in range(1,11)], rotation=0)
plt.tight_layout()
plt.show()
y_raw_full = ts_df['safety_bin_1_10']

# 3. Final Merge and Alignment
combined_df = X_df.merge(y_raw_full, left_index=True, right_index=True, how='inner')
target_col_name = [col for col in combined_df.columns if 'safety_bin_1_10' in col][-1]

X = combined_df.drop(columns=[target_col_name]).values
y_raw = combined_df[target_col_name].values.astype(int)  # ensure int
y_raw = y_raw.astype(int) - 1
# 4. Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\n[4/8] Features scaled using StandardScaler.")

# --- 5-Fold Stratified Cross-Validation ---
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Storage for aggregated results
fold_histories = []
fold_val_acc = []
fold_val_loss = []
y_true_all = []
y_pred_all = []
total_cm = np.zeros((TOTAL_CLASSES, TOTAL_CLASSES), dtype=int)

def create_robust_model(input_dim, output_dim):
    """Creates a deeper, more robust feed-forward neural network."""
    model = Sequential([
        Dense(256, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(output_dim, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

fold_idx = 1
for train_index, val_index in skf.split(X_scaled, y_raw):
    print(f"\n--- Fold {fold_idx}/{N_SPLITS} ---")
    # clear keras session & create fresh model each fold
    K.clear_session()

    X_train, X_val = X_scaled[train_index], X_scaled[val_index]
    y_train_raw, y_val_raw = y_raw[train_index], y_raw[val_index]

    # One-hot encode targets (keep consistent with TOTAL_CLASSES)
    y_train = to_categorical(y_train_raw, num_classes=TOTAL_CLASSES)
    y_val = to_categorical(y_val_raw, num_classes=TOTAL_CLASSES)

    # Class weights computed from training fold (only classes present in this fold)
    present_classes = np.unique(y_train_raw)
    cw_array = compute_class_weight('balanced', classes=present_classes, y=y_train_raw)
    class_weight_dict = {int(c): w for c, w in zip(present_classes, cw_array)}
    print("Class weights for this fold (only present classes shown):", class_weight_dict)

    # Callbacks: make new instances for each fold
    early_stop = EarlyStopping(monitor='val_loss', patience=10, verbose=1, mode='min', restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
    callbacks = [early_stop, reduce_lr]

    # Create and train the model
    input_dim = X_train.shape[1]
    output_dim = TOTAL_CLASSES
    model = create_robust_model(input_dim, output_dim)
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, y_val),
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )
    # --- Plot Training Curves for This Fold ---
    epochs = range(1, len(history.history['loss']) + 1)

    plt.figure(figsize=(14, 6))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history['accuracy'], label='Training Accuracy')
    plt.plot(epochs, history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title(f'Fold {fold_idx} - Accuracy')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history['loss'], label='Training Loss')
    plt.plot(epochs, history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Fold {fold_idx} - Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()


    # Store fold history and metrics
    fold_histories.append(history)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    fold_val_acc.append(val_acc)
    fold_val_loss.append(val_loss)
    print(f"Fold {fold_idx} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Predictions for this fold (use argmax indices)
    y_val_pred_probs = model.predict(X_val)
    y_val_pred = np.argmax(y_val_pred_probs, axis=1)
    y_val_true = np.array(y_val_raw, dtype=int)

    # Accumulate true/pred for global report and confusion matrix (force full shape)
    y_true_all.extend(y_val_true.tolist())
    y_pred_all.extend(y_val_pred.tolist())

    cm_fold = confusion_matrix(y_val_true, y_val_pred, labels=range(TOTAL_CLASSES))
    total_cm += cm_fold

    fold_idx += 1

# --- After CV: Aggregate Results ---
print("\n--- Cross-Validation Summary ---")
for i, (vl, va) in enumerate(zip(fold_val_loss, fold_val_acc), start=1):
    print(f"Fold {i}: Val Loss = {vl:.4f}, Val Acc = {va:.4f}")
print(f"\nMean Val Accuracy: {np.mean(fold_val_acc):.4f} ± {np.std(fold_val_acc):.4f}")
print(f"Mean Val Loss: {np.mean(fold_val_loss):.4f} ± {np.std(fold_val_loss):.4f}")

# Overall classification report on aggregated fold predictions
target_names = [f'Bin {i}' for i in range(1, TOTAL_CLASSES + 1)]
print("\n--- Aggregated Classification Report (all validation folds) ---")
print(classification_report(y_true_all, y_pred_all, target_names=target_names, labels=range(TOTAL_CLASSES), zero_division=0))

# Plot aggregated confusion matrix (sum across folds)
print("\n--- Aggregated Confusion Matrix (sum across folds) ---")
fig, ax = plt.subplots(figsize=(max(6, TOTAL_CLASSES), max(6, TOTAL_CLASSES)))
disp = ConfusionMatrixDisplay(confusion_matrix=total_cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
ax.set_title('Aggregated Confusion Matrix (5-fold CV)', fontsize=16)
plt.tight_layout()
plt.show()

print("\n--- 5-Fold Cross-Validation Complete ---")
model.save("my_model.keras")



In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

from IPython.display import HTML

# --- Configuration ---
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
MODEL_FILE = "my_model.keras"
# Base URL for public image access
BASE_IMAGE_URL = "https://cycleai.net/london/training/training/Images/"
TOTAL_CLASSES = 10

print("--- Starting Inference Pipeline ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['ai_ts_score']].rename(columns={'ai_ts_score': 'safety_score'})
except FileNotFoundError:
    print("Error: Input CSV files not found.")
    exit()

print(f"Diagnostic: Feature file (X_df) loaded with {len(X_df)} images.")
print(f"Diagnostic: TrueSkill file (ts_df) loaded with {len(ts_df)} scores.")

# 2. Replicate Training Processing (Crucial for consistency)
# We must perform the exact same binning to know what "Bin 0", "Bin 1", etc. represent.
try:
    # Calculate bins using qcut with duplicates='drop' (MATCHING TRAINING CODE)
    # We need retbins=True to retrieve the specific edges for score estimation later
    ts_df['safety_bin'], bin_edges = pd.qcut(
        ts_df['safety_score'],
        q=TOTAL_CLASSES,
        labels=False,
        duplicates='drop',
        retbins=True
    )
    print(f"Recovered {len(bin_edges)-1} bins from training logic (qcut).")
    print(f"Bin edges: {np.round(bin_edges, 3)}")

except ValueError as e:
    print(f"Error reproducing binning logic: {e}")
    exit()

# 3. Fit Scaler on Training Data
# The scaler must be fit on the exact same data used to train the model.
# The inner merge finds the set of images that have both features AND valid labels.
combined_df = X_df.merge(ts_df[['safety_bin']], left_index=True, right_index=True, how='inner')

# Training features (X) used for fitting the scaler
X_train_raw = combined_df.drop(columns=['safety_bin']).values

scaler = StandardScaler()
scaler.fit(X_train_raw)
print(f"Scaler re-fitted on {len(X_train_raw)} labeled images (combined_df size).")

# 4. Select 10 Images for Prediction
# Unseen = Images in original Feature File (X_df) that were NOT part of the final training set (combined_df)
unseen_indices = X_df.index.difference(combined_df.index)

print(f"Diagnostic: Found {len(unseen_indices)} images that are in the feature file but not the training set.")

if len(unseen_indices) == 0:
    print("\nWARNING: No new/unseen images found. This means all images in your feature file also have a score.")
    print("Falling back to selecting 10 random images from the full feature set (X_df) to demonstrate the pipeline.")
    sample_indices = np.random.choice(X_df.index, size=10, replace=False)
elif len(unseen_indices) < 10:
    print(f"Not enough unseen images found (only {len(unseen_indices)}). Prediction will be run on these.")
    sample_indices = unseen_indices
else:
    # Randomly sample 10 unseen images
    sample_indices = np.random.choice(unseen_indices, size=10, replace=False)

print(f"\nSelected 10 images for prediction: {sample_indices}")

# 5. Prepare Features for the Sample
X_sample_raw = X_df.loc[sample_indices].values
X_sample_scaled = scaler.transform(X_sample_raw)

# 6. Load Model and Predict
try:
    if not os.path.exists(MODEL_FILE):
        print(f"Model file '{MODEL_FILE}' not found. Please ensure the training script ran successfully.")
        exit()

    model = load_model(MODEL_FILE)
    predictions_probs = model.predict(X_sample_scaled, verbose=0)

    # Get the predicted bin index
    predicted_bins = np.argmax(predictions_probs, axis=1)

    # Calculate Estimated Score based on qcut edges
    # We take the midpoint of the predicted bin's edges
    estimated_scores = []
    for bin_idx in predicted_bins:
        # Safety check: ensure predicted bin is within the edges we recovered
        if bin_idx < len(bin_edges) - 1:
            lower = bin_edges[bin_idx]
            upper = bin_edges[bin_idx + 1]
            midpoint = (lower + upper) / 2
            estimated_scores.append(midpoint)
        else:
            # Fallback for out-of-bounds prediction (e.g., if model predicts bin 9 but only 6 bins were created)
            print(f"Warning: Predicted bin {bin_idx} is outside the known range {len(bin_edges)-1}. Using max edge.")
            estimated_scores.append(bin_edges[-1])

except Exception as e:
    print(f"Error during inference: {e}")
    exit()

# 7. Display Results with Image URL in a formatted HTML table
print("\n--- Prediction Results (Visual Display) ---")

html_content = """
<style>
    .prediction-table { border-collapse: collapse; width: 100%; font-family: sans-serif; }
    .prediction-table th, .prediction-table td { border: 1px solid #ddd; padding: 8px; text-align: left; }
    .prediction-table th { background-color: #f2f2f2; color: #333; }
    .prediction-image { max-width: 150px; height: auto; border-radius: 4px; }
    .score-container { display: flex; align-items: center; }
</style>
<table class="prediction-table">
    <thead>
        <tr>
            <th>Image Name</th>
            <th>Predicted Bin</th>
            <th>Est. Safety Score</th>
            <th>Image Preview</th>
        </tr>
    </thead>
    <tbody>
"""

for i, img_name in enumerate(sample_indices):
    bin_idx = predicted_bins[i]
    score = estimated_scores[i]
    image_url = f"{BASE_IMAGE_URL}{img_name}"

    # Calculate color for visual representation (normalized 0 to 100, then mapped to a green/red gradient)
    # This is a heuristic mapping for visualization
    # We use a simple linear interpolation based on the score bounds recovered from qcut
    min_score = bin_edges[0]
    max_score = bin_edges[-1]

    # Normalize score between 0 and 1
    if max_score > min_score:
        normalized_score = (score - min_score) / (max_score - min_score)
    else:
        normalized_score = 0.5

    # Interpolate from Red (0, unsafe) to Green (1, safe)
    red = int(255 * (1 - normalized_score))
    green = int(255 * normalized_score)
    score_color = f"rgb({red}, {green}, 0)"

    html_content += f"""
        <tr>
            <td>{img_name}</td>
            <td>Bin {bin_idx}</td>
            <td>
                <div class="score-container">
                    <span style="background-color: {score_color}; color: white; padding: 3px 6px; border-radius: 4px; margin-right: 10px; font-weight: bold;">
                        {score:.4f}
                    </span>
                    <small>(Range: {bin_edges[bin_idx]:.2f} - {bin_edges[bin_idx+1]:.2f})</small>
                </div>
            </td>
            <td>
                <a href="{image_url}" target="_blank">
                    <img src="{image_url}" class="prediction-image" alt="Preview of {img_name}">
                </a>
            </td>
        </tr>
    """

html_content += """
    </tbody>
</table>
<p>(Note: 'Est. Safety Score' is the midpoint of the Quantile Bin from training. Click the image to view full size.)</p>
"""
display(HTML(html_content))